# Build the feature CSV — 5 sampled videos

Takes the AEA download-links JSON, **samples 5 videos**, runs the pipeline end to end on
each, and writes **one combined CSV** with a row per consecutive frame pair.

All pipeline code is borrowed **verbatim** from the repo — `aria_io.py`,
`gaze_geometry.py`, `encoders.py` — so no clone is needed.

## Output columns

| Column | Meaning |
|---|---|
| `idx` | **global** row index, 0 … N−1 across all videos |
| `sequence` | which video this row came from |
| `feat_frame_1` | path to the saved DINOv2 features of frame *t−1* |
| `feat_frame_2` | path to the saved DINOv2 features of frame *t* |
| `gaze_patch_token_sim` | cosine of the gaze patch tokens |
| `frame_similarity` | cosine of the whole-frame CLS tokens |
| `n_velocities` | number of steps in the window (9 at 10 Hz) |
| `gaze_rates_window` | the **9 × 3** array: `omega_yaw, omega_pitch, omega_mag` per step |

## Structure

Sections 5–8 only **define** the borrowed functions. Section 9 holds the single loop that
runs all of them over each sampled video and accumulates into one table. Everything after
that is display and verification, unchanged.

## Expected row count

With `MAX_SECONDS = 90` at 1 FPS: 90 frames per video → **89 pairs each** → **445 rows**
for 5 videos. Any video shorter than 90 s contributes fewer; the loop prints the actual
per-sequence and total counts.

## Cost

The VRS is required for the calibration projection, so this downloads roughly **12 GB**
across 5 videos. Set `CLEANUP_RAW = True` in section 4 to delete each video's raw files
once its features are written, which caps peak disk at about one video.

**No GPU needed** — DINOv2 on ~450 frames runs in under a minute on CPU."

## 1 — Setup

In [ ]:
!pip -q install opencv-python-headless pandas numpy projectaria-tools

import os, glob, json, zipfile, urllib.request, time, random
import numpy as np, pandas as pd, cv2
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("opencv", cv2.__version__, "| torch", torch.__version__, "| device", DEV)

## 2 — Load the download-links JSON

In [ ]:
from google.colab import files
up = files.upload()                 # pick your aea_download_urls.json
URLS_JSON = list(up.keys())[0]

# or, if it already lives in Drive:
# from google.colab import drive; drive.mount('/content/drive')
# URLS_JSON = "/content/drive/MyDrive/aea/aea_download_urls.json"

meta = json.load(open(URLS_JSON))
seqs = list(meta["sequences"])
print(f"{len(seqs)} videos in the JSON")

## 3 — Sample the videos

`random.sample` draws **without replacement**, so no video is picked twice. Change
`N_VIDEOS` or `SEED` to get a different draw.

In [ ]:
N_VIDEOS = 5
SEED     = 0

random.seed(SEED)
SEQS = random.sample(seqs, N_VIDEOS)      # sample WITHOUT replacement

# to pin specific ones instead, uncomment:
# SEQS = ["loc5_script4_seq6_rec1", "loc4_script1_seq1_rec1", "loc3_script5_seq6_rec1"]

print(f"sampled {len(SEQS)} of {len(seqs)} videos (seed {SEED}):\n")
grand = 0
for i, s in enumerate(SEQS, 1):
    ee = meta["sequences"][s]
    for k in ("video_main_rgb", "mps_eye_gaze", "main_vrs"):
        assert k in ee, f"{s}: missing required entry {k}"
    tot = sum(ee[k]["file_size_bytes"] for k in ("video_main_rgb", "mps_eye_gaze", "main_vrs"))
    grand += tot
    print(f"   {i}. {s:28s} {tot/1e6:8,.0f} MB")

print(f"\n   TOTAL DOWNLOAD: {grand/1e9:.1f} GB")
import shutil
free = shutil.disk_usage("/content").free / 1e9
print(f"   free disk     : {free:.1f} GB   {'OK' if free > grand/1e9 + 3 else '!! LOW -- set CLEANUP_RAW = True'}")

## 4 — Configure

In [ ]:
RAW_DIR     = "/content/build/raw"
FRAMES_DIR  = "/content/build/frames_1fps"
FEAT_DIR    = "/content/build/features"      # <- the .npz feature files land here
OUT_CSV     = "/content/build/feature_dataset.csv"

TARGET_FPS  = 1.0
MAX_SECONDS = None      # None = the WHOLE video. Set a number only to trim a debug run:
                        # capping discards footage the download has already paid for.
DINO_SIZE   = 98        # matches configs/temporal_analysis.yaml -> dino_input_patch
DEPTH_M     = 1.0       # configs -> gaze_depth_m
ROTATE_CW90 = True      # configs -> rotate_cw90 (preview mp4 is upright)

CLEANUP_RAW = False     # True -> delete each sequence's mp4/vrs after processing it,
                        # so peak disk stays at ~1 sequence instead of N_VIDEOS

for d in (RAW_DIR, FRAMES_DIR, FEAT_DIR, os.path.dirname(OUT_CSV)):
    os.makedirs(d, exist_ok=True)

print(f"videos     : {len(SEQS)}")
print(f"output CSV : {OUT_CSV}")
print(f"features   : {FEAT_DIR}/<sequence>/")
if MAX_SECONDS:
    exp = int(MAX_SECONDS * TARGET_FPS)
    print(f"\ntrimmed to {MAX_SECONDS}s: ~{exp} frames per video -> ~{exp-1} rows each")
    print(f"          ~{(exp-1)*len(SEQS)} rows total")
else:
    print(f"\nwhole videos: AEA averages ~193 s (67-456 s), so at {TARGET_FPS} FPS")
    print(f"          expect ~190 rows per video, ~{190*len(SEQS)} rows total")

## 5 — Stage 0: download

`download_sequence` verbatim from `aria_io.py:5-37`, progress hook added.

In [ ]:
def _progress(name):
    t0 = time.time()
    def hook(blocks, bs, total):
        done = blocks * bs
        pct  = 100 * done / total if total > 0 else 0
        print(f"\r   {name:16s} {done/1e6:8.1f} MB  {min(pct,100):5.1f}%  ({time.time()-t0:4.0f}s)", end="")
    return hook


def download_sequence(seq, meta, raw_dir, want_calibration=False):
    seq_dir = os.path.join(raw_dir, seq)
    os.makedirs(os.path.join(seq_dir, "eye_gaze"), exist_ok=True)
    e = meta["sequences"][seq]

    rgb = e["video_main_rgb"]
    mp4 = os.path.join(seq_dir, rgb["filename"])
    if not os.path.exists(mp4):
        urllib.request.urlretrieve(rgb["download_url"], mp4, _progress("video_main_rgb")); print()

    eg = e["mps_eye_gaze"]
    zp = os.path.join(seq_dir, eg["filename"])
    if not os.path.exists(zp):
        urllib.request.urlretrieve(eg["download_url"], zp, _progress("mps_eye_gaze")); print()
    with zipfile.ZipFile(zp) as z:
        z.extractall(os.path.join(seq_dir, "eye_gaze"))

    vrs_path = None
    if want_calibration:
        vrs = e.get("main_vrs")
        if vrs is None:
            raise RuntimeError(f"[{seq}] calibration/IMU requested but no 'main_vrs' entry.")
        vrs_path = os.path.join(seq_dir, vrs["filename"])
        if not os.path.exists(vrs_path):
            urllib.request.urlretrieve(vrs["download_url"], vrs_path, _progress("main_vrs")); print()
        if not os.path.exists(vrs_path):
            raise RuntimeError(f"[{seq}] VRS download failed; no file at {vrs_path}")

    return seq_dir, mp4, vrs_path


print("download_sequence defined (runs inside the loop in section 9)")

## 6 — Stage 1: frames at 1 FPS, and the gaze stream

`subsample_frames` and `load_gaze_raw` verbatim from `aria_io.py`.

In [ ]:
def subsample_frames(mp4, out_dir, target_fps=1.0, max_frames=None):
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(mp4); native = cap.get(cv2.CAP_PROP_FPS) or 20.0
    step = max(1, int(round(native / target_fps)))
    paths, ts_us, kept, i = [], [], 0, 0
    while True:
        ok, fr = cap.read()
        if not ok or (max_frames is not None and i >= max_frames):
            break
        if i % step == 0:
            p = os.path.join(out_dir, f"frame_{kept:05d}.jpg")
            cv2.imwrite(p, fr)
            paths.append(p); ts_us.append((i / native) * 1e6); kept += 1
        i += 1
    cap.release()
    return paths, np.array(ts_us), native


def load_gaze_raw(seq_dir):
    gcsv = glob.glob(os.path.join(seq_dir, "eye_gaze", "**", "general_eye_gaze.csv"),
                     recursive=True)[0]
    g = pd.read_csv(gcsv)
    ts = g["tracking_timestamp_us"].to_numpy(); ts = ts - ts[0]
    return ts, g["yaw_rads_cpf"].to_numpy(), g["pitch_rads_cpf"].to_numpy()


print("subsample_frames + load_gaze_raw defined (run inside the loop in section 9)")

## 7 — The gaze projector

`make_gaze_projector` verbatim from `gaze_geometry.py:111-144` — the official Aria
transform chain plus the native-to-upright rotation.

In [ ]:
from projectaria_tools.core import data_provider
from projectaria_tools.core.mps import get_eyegaze_point_at_depth

def native_to_upright_cw90(x, y):
    return 1.0 - y, x


def make_gaze_projector(vrs_path, stream_label="camera-rgb", depth_m=1.0, rotate_cw90=True):
    provider     = data_provider.create_vrs_data_provider(vrs_path)
    device_calib = provider.get_device_calibration()
    cam_calib    = device_calib.get_camera_calib(stream_label)
    T_device_cpf = device_calib.get_transform_device_cpf()
    T_device_cam = cam_calib.get_transform_device_camera()
    W, H = cam_calib.get_image_size()

    def _proj(yaw, pitch, depth=depth_m):
        gaze_cpf = get_eyegaze_point_at_depth(yaw, pitch, depth)
        gaze_cam = T_device_cam.inverse() @ T_device_cpf @ gaze_cpf
        px = cam_calib.project(gaze_cam)
        if px is None:
            return 0.5, 0.5, True
        u, v = np.asarray(px, dtype=np.float64).flatten()[:2]
        x, y = u / W, v / H
        if rotate_cw90:
            x, y = native_to_upright_cw90(x, y)
        return float(np.clip(x, 0, 1)), float(np.clip(y, 0, 1)), False

    return _proj


# NOTE: a projector is built PER SEQUENCE inside the loop -- the calibration is
# specific to the physical pair of glasses that recorded it, so it cannot be shared.
print("make_gaze_projector defined (one projector per sequence, built in section 9)")

## 8 — Stage 2: encode with DINOv2 and **save the features**

`frame_features` verbatim from `encoders.py:12-21`. One `.npz` per frame containing:

| key | shape | what |
|---|---|---|
| `cls` | (384,) | whole-frame token, L2-normalized |
| `patches` | (49, 384) | the 7×7 grid of patch tokens, L2-normalized |
| `grid` | scalar | 7 |
| `gaze_xy` | (2,) | projected gaze on this frame |
| `t_us` | scalar | frame timestamp |

Self-contained, so a training loader needs nothing but these files.

In [ ]:
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(DEV).eval()
for p_ in dino.parameters():
    p_.requires_grad_(False)


@torch.no_grad()
def frame_features(frame_rgb, device, dino_size=DINO_SIZE):
    """SINGLE DINOv2 pass -> (cls_token, patch_tokens_grid, grid)."""
    t = torch.from_numpy(frame_rgb.copy()).permute(2,0,1).float().unsqueeze(0).to(device)/255.0
    t = F.interpolate(t, size=(dino_size, dino_size), mode="bilinear", align_corners=False)
    out = dino.forward_features(t)
    cls = F.normalize(out["x_norm_clstoken"][0], dim=-1)
    patches = F.normalize(out["x_norm_patchtokens"][0], dim=-1)
    grid = int(round(patches.shape[0] ** 0.5))
    return cls, patches, grid


def encode_and_save(seq, paths, f_ts, g_ts, g_yaw, g_pit, project):
    """Encode every frame once, write one .npz per frame, return the in-memory tensors
    plus the file paths."""
    out_dir = os.path.join(FEAT_DIR, seq)
    os.makedirs(out_dir, exist_ok=True)

    cls_l, pat_l, gxy, feat_paths, oof = [], [], [], [], 0
    grid = None
    for k, (p_, t_) in enumerate(zip(paths, f_ts)):
        frame = cv2.cvtColor(cv2.imread(p_), cv2.COLOR_BGR2RGB)
        j = int(np.clip(np.searchsorted(g_ts, t_), 0, len(g_ts) - 1))
        o = project(g_yaw[j], g_pit[j])
        x_, y_, o_ = float(o[0]), float(o[1]), bool(o[2])
        oof += int(o_)

        cls, patches, grid = frame_features(frame, DEV)

        fp = os.path.join(out_dir, f"feat_{k:05d}.npz")
        np.savez_compressed(
            fp,
            cls=cls.cpu().numpy().astype(np.float32),
            patches=patches.cpu().numpy().astype(np.float32),
            grid=np.int32(grid),
            gaze_xy=np.array([x_, y_], dtype=np.float32),
            t_us=np.float64(t_),
        )
        feat_paths.append(fp)
        cls_l.append(cls); pat_l.append(patches); gxy.append((x_, y_, o_))

    return cls_l, pat_l, gxy, feat_paths, grid, oof


print("DINOv2 loaded + encode_and_save defined (runs inside the loop in section 9)")

## 9 — Run everything, for every sampled video

`patch_token_at_gaze` / `cosine` from `encoders.py`, `window_gaze_rates` from
`gaze_geometry.py:32-64` — then the **main loop**.

Per video it downloads, subsamples to 1 FPS, loads gaze, builds a projector, encodes and
saves features, and appends one row per consecutive frame pair. Rows accumulate into a
single `rows` list, so `idx` runs **globally** from 0 to N−1 rather than restarting per
video.

Two things are deliberately per-sequence:

- **the projector** — the calibration belongs to the physical pair of glasses that made
  the recording, so it cannot be reused across videos
- **the feature folder** — `features/<sequence>/feat_00000.npz`, keeping paths unambiguous

This is the slow cell: expect a few minutes, dominated by the VRS downloads.

In [ ]:
def patch_token_at_gaze(patches, grid, gaze_xy_norm):
    gx = min(grid-1, int(np.clip(gaze_xy_norm[0], 0, 1) * grid))
    gy = min(grid-1, int(np.clip(gaze_xy_norm[1], 0, 1) * grid))
    return patches[gy*grid + gx]

def cosine(a, b):
    return float(F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item())


def window_gaze_rates(g_ts_us, g_yaw, g_pit, t0_us, t1_us):
    """(n-1, 3) signal: omega_yaw, omega_pitch, omega_mag  (rad/s)."""
    m = (g_ts_us >= t0_us) & (g_ts_us < t1_us)
    ts, ya, pi = g_ts_us[m], g_yaw[m], g_pit[m]
    good = np.isfinite(ya) & np.isfinite(pi)
    ts, ya, pi = ts[good], ya[good], pi[good]
    if len(ts) < 2:
        return dict(rates=np.zeros((0, 3)), n_samples=int(len(ts)), n_velocities=0)
    dts = np.diff(ts) / 1e6
    dts = np.where(dts <= 0, 1e-6, dts)
    w_yaw = np.diff(ya) / dts
    w_pit = np.diff(pi) / dts
    w_mag = np.hypot(w_yaw, w_pit)
    return dict(rates=np.stack([w_yaw, w_pit, w_mag], axis=1),
                n_samples=int(len(ts)), n_velocities=int(len(dts)))


# =====================================================================================
#  MAIN LOOP -- every stage, for every sampled video, accumulating into one `rows` list
# =====================================================================================
rows, per_seq = [], []
GRID = None
t_all = time.time()

for si, SEQ in enumerate(SEQS, 1):
    print(f"\n{'='*80}\n[{si}/{len(SEQS)}]  {SEQ}\n{'='*80}")
    t_seq = time.time()

    # -- Stage 0: download -------------------------------------------------------
    seq_dir, mp4, vrs_path = download_sequence(SEQ, meta, RAW_DIR, want_calibration=True)

    # -- Stage 1: frames + gaze --------------------------------------------------
    cap = cv2.VideoCapture(mp4); native = cap.get(cv2.CAP_PROP_FPS) or 20.0; cap.release()
    mf = int(MAX_SECONDS * native) if MAX_SECONDS else None
    paths, f_ts, native = subsample_frames(mp4, os.path.join(FRAMES_DIR, SEQ), TARGET_FPS, mf)
    g_ts, g_yaw, g_pit = load_gaze_raw(seq_dir)
    print(f"   frames {len(paths)}   gaze {len(g_ts):,} @ "
          f"{1/np.median(np.diff(g_ts)/1e6):.1f} Hz over {g_ts[-1]/1e6:.0f} s")

    # -- projector (per sequence: calibration is device-specific) ----------------
    project = make_gaze_projector(vrs_path, depth_m=DEPTH_M, rotate_cw90=ROTATE_CW90)

    # -- Stage 2: encode + save features -----------------------------------------
    cls_l, pat_l, gxy, feat_paths, GRID, oof = encode_and_save(
        SEQ, paths, f_ts, g_ts, g_yaw, g_pit, project)

    # -- Stages 3 + 4: similarities and rates ------------------------------------
    n_before = len(rows)
    for i in range(1, len(paths)):
        frame_sim = cosine(cls_l[i-1], cls_l[i])
        pt0 = patch_token_at_gaze(pat_l[i-1], GRID, gxy[i-1][:2])
        pt1 = patch_token_at_gaze(pat_l[i],   GRID, gxy[i][:2])
        patch_sim = cosine(pt0, pt1)

        rr = window_gaze_rates(g_ts, g_yaw, g_pit, f_ts[i-1], f_ts[i])

        rows.append(dict(
            idx=len(rows),                       # GLOBAL index across all videos
            sequence=SEQ,
            feat_frame_1=feat_paths[i-1],
            feat_frame_2=feat_paths[i],
            gaze_patch_token_sim=round(patch_sim, 4),
            frame_similarity=round(frame_sim, 4),
            n_velocities=rr["n_velocities"],
            gaze_rates_window=";".join(f"{r[0]:.5f},{r[1]:.5f},{r[2]:.5f}" for r in rr["rates"]),
        ))

    n_rows = len(rows) - n_before
    feat_mb = sum(os.path.getsize(f) for f in feat_paths) / 1e6
    per_seq.append(dict(sequence=SEQ, frames=len(paths), rows=n_rows,
                        oof_frames=oof, feat_MB=round(feat_mb, 1),
                        secs=round(time.time()-t_seq, 1)))
    print(f"   -> {n_rows} rows   (running total {len(rows)})   "
          f"features {feat_mb:.1f} MB   {time.time()-t_seq:.0f}s")

    if CLEANUP_RAW:
        shutil.rmtree(seq_dir, ignore_errors=True)
        shutil.rmtree(os.path.join(FRAMES_DIR, SEQ), ignore_errors=True)
        print(f"   cleaned up raw + frames for {SEQ}")

    del cls_l, pat_l
    if DEV == "cuda":
        torch.cuda.empty_cache()

# =====================================================================================
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print(f"\n{'='*80}\nDONE in {time.time()-t_all:.0f}s\n{'='*80}")
print(f"\nper sequence:")
display(pd.DataFrame(per_seq))
print(f"\nTOTAL ROWS (idx 0 .. {len(df)-1})  =  {len(df)}")
print(f"   videos     : {df['sequence'].nunique()}")
print(f"   columns    : {df.shape[1]}")
print(f"   CSV        : {OUT_CSV}  ({os.path.getsize(OUT_CSV)/1e3:.0f} KB)")
print(f"   idx unique : {df['idx'].is_unique}   contiguous: {list(df['idx']) == list(range(len(df)))}")

## 10 — The DataFrame

In [ ]:
pd.set_option("display.max_colwidth", 40)

print("=" * 100)
print(f"DATAFRAME — {len(df)} rows from {df['sequence'].nunique()} videos")
print("   (paths shortened to basenames, rate strings truncated, for readability)")
print("=" * 100)

df_show = df.copy()
df_show["feat_frame_1"] = df_show["feat_frame_1"].apply(os.path.basename)
df_show["feat_frame_2"] = df_show["feat_frame_2"].apply(os.path.basename)
df_show["sequence"] = df_show["sequence"].str.slice(0, 22)
df_show["gaze_rates_window"] = df_show["gaze_rates_window"].str.slice(0, 30) + " ..."

print("\nhead:")
display(df_show.head(8))
print("tail:")
display(df_show.tail(8))

print("\nrows per sequence (note idx is GLOBAL, so it does not restart):")
display(df.groupby("sequence").agg(
    rows=("idx", "size"),
    idx_from=("idx", "min"),
    idx_to=("idx", "max"),
    frame_sim_mean=("frame_similarity", "mean"),
    gaze_sim_mean=("gaze_patch_token_sim", "mean"),
).round(3))

print("\ndtypes and non-null counts:")
df.info()

print("\nnumeric summary across all videos:")
display(df[["gaze_patch_token_sim", "frame_similarity", "n_velocities"]].describe().round(4))

print(f"\nTOTAL idx COUNT = {len(df)}   (idx 0 .. {len(df)-1})")

print("\nfull first row, every column:")
for c in df.columns:
    v = str(df.loc[0, c])
    print(f"   {c:22s} {v[:110]}{' ...' if len(v) > 110 else ''}")

## 11 — Verify: the saved features are usable on their own

Reloads two `.npz` files **from disk** and recomputes both similarities from them alone.
If these match the CSV, a training loader can use the feature files without ever touching
the video, the VRS, or DINOv2 again.

In [ ]:
ROW = 0
z0 = np.load(df.loc[ROW, "feat_frame_1"])
z1 = np.load(df.loc[ROW, "feat_frame_2"])

print(f"contents of {os.path.basename(df.loc[ROW,'feat_frame_1'])}:")
for k in z0.files:
    a = z0[k]
    print(f"   {k:10s} shape={str(a.shape):12s} dtype={a.dtype}")

# tokens are already L2-normalized, so a dot product IS the cosine
fs = float(np.dot(z0["cls"], z1["cls"]))

g = int(z0["grid"])
def cell(zz):
    x, y = zz["gaze_xy"]
    gx = min(g-1, int(np.clip(x, 0, 1) * g)); gy = min(g-1, int(np.clip(y, 0, 1) * g))
    return zz["patches"][gy*g + gx]
ps = float(np.dot(cell(z0), cell(z1)))

print(f"\nrecomputed FROM THE FILES vs what the CSV says:")
print(f"   frame_similarity     : {fs:.4f}   vs   {df.loc[ROW,'frame_similarity']:.4f}")
print(f"   gaze_patch_token_sim : {ps:.4f}   vs   {df.loc[ROW,'gaze_patch_token_sim']:.4f}")
ok = (abs(fs - df.loc[ROW,'frame_similarity']) < 1e-3 and
      abs(ps - df.loc[ROW,'gaze_patch_token_sim']) < 1e-3)
print(f"\n   [{'PASS' if ok else 'FAIL'}]  feature files round-trip correctly")

# unpack the packed rate string back into its (9, 3) array
R = np.array([[float(v) for v in trip.split(",")]
              for trip in df.loc[ROW, "gaze_rates_window"].split(";")])
print(f"\n   gaze_rates_window unpacks to shape {R.shape}   (steps x [yaw, pitch, mag])")
print("   step |  omega_yaw   omega_pitch   omega_mag")
print("   -----+---------------------------------------")
for k, r in enumerate(R):
    print(f"   {k:4d} | {r[0]:+10.4f}  {r[1]:+11.4f}  {r[2]:10.4f}")
print(f"\n   [{'PASS' if R.shape == (df.loc[ROW,'n_velocities'], 3) else 'FAIL'}]  "
      f"shape matches n_velocities = {df.loc[ROW,'n_velocities']}")

## 12 — Quick look at the two label distributions

In [ ]:
mags = np.array([np.mean([float(t.split(",")[2]) for t in s.split(";")])
                 for s in df["gaze_rates_window"]])

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].hist(df["frame_similarity"], bins=30, alpha=.7, label="frame_similarity")
ax[0].hist(df["gaze_patch_token_sim"], bins=30, alpha=.7, label="gaze_patch_token_sim")
ax[0].set_xlabel("cosine similarity"); ax[0].legend(); ax[0].set_title("the two teacher labels")

ax[1].scatter(df["frame_similarity"], df["gaze_patch_token_sim"], s=20, alpha=.7)
ax[1].set_xlabel("frame_similarity"); ax[1].set_ylabel("gaze_patch_token_sim")
r = np.corrcoef(df["frame_similarity"], df["gaze_patch_token_sim"])[0,1]
ax[1].set_title(f"correlation {r:+.3f}")

ax[2].scatter(mags, df["frame_similarity"], s=20, alpha=.7)
ax[2].set_xlabel("mean omega_mag in window (rad/s)"); ax[2].set_ylabel("frame_similarity")
r2 = np.corrcoef(mags, df["frame_similarity"])[0,1]
ax[2].set_title(f"THE PREMISE: gaze speed vs frame change   r = {r2:+.3f}")
plt.tight_layout(); plt.show()

print(f"gaze speed vs frame_similarity correlation: {r2:+.3f}")
print("   NEGATIVE is what the project predicts -- faster eye movement should mean")
print("   a bigger visual change, hence LOWER similarity.")
print("   Near zero on one 90 s clip is not yet evidence against it; check across sequences.")

---

## What you end up with

```
/content/build/
├── raw/<SEQ>/                 mp4 + gaze csv + vrs      (~2.5 GB each, discardable)
├── frames_1fps/<SEQ>/         frame_00000.jpg ...       (discardable)
├── features/<SEQ>/            feat_00000.npz ...        <- KEEP THIS
└── feature_dataset.csv        one table, all videos     <- KEEP THIS
```

Only the last two matter for training. Everything above can be deleted once the CSV is
written — which is the point of storing feature paths rather than JPEG paths, and what
`CLEANUP_RAW = True` automates during the run.

## Row count

At `MAX_SECONDS = 90` and 1 FPS, each video yields 90 frames → **89 rows**, so 5 videos
give **445**. `idx` is global and contiguous, 0 … 444; the loop asserts both at the end.
A video shorter than 90 s contributes fewer rows, so check the per-sequence table if the
total comes in under 445.

## To scale up

- **more videos** — raise `N_VIDEOS`. Downloads grow ~2.5 GB each, so turn on
  `CLEANUP_RAW`.
- **longer clips** — raise `MAX_SECONDS` or set it to `None`. Rows grow roughly one per
  second of video; a full ~213 s sequence gives ~212 rows instead of 89.
- **re-running** — already-downloaded files are skipped by the `os.path.exists` guards, so
  a re-run only redoes the encoding.

## Caveats worth carrying forward

- **`DEPTH_M = 1.0` is a fixed assumption.** The MPS gaze file ships a per-sample `depth_m`
  column that is not used here. Objects held in the hands sit closer to 0.4 m, and the
  ~6 cm eye-to-camera baseline makes that mismatch visible in the gaze dot.
- **The 7×7 grid is coarse** — one patch token covers ~200×200 px of a 1408 px frame.
  Raising `DINO_SIZE` to 224 gives 16×16 = 256 tokens for roughly 5× the encode time.
- **`omega_mag` is not sphere-exact.** Yaw and pitch are not orthogonal on a sphere, so the
  yaw term is overstated by `1/cos(pitch)`. Deliberate: the 3-channel form is what the
  gating CNN consumes.
- **Each video has its own calibration.** The projector is rebuilt per sequence for that
  reason; do not hoist it out of the loop.